In [1]:
from weak_stage_utils import *

# First, we do training using the weak labelled dataset PQA-A

weak_path = "../Preprocessing+baseline/data/ori_pqaa.json"

full_split_save_path = "./data"

tokenizer = get_tokenizer()

weak_dataset = load_weak_json(weak_path)
validate_weak_dataset(weak_dataset)

weak_dataset = weak_dataset.map(add_binary_label)

full_split = build_full_split(weak_dataset, val_size=0.1, seed=7)
save_dataset_split(full_split, full_split_save_path)

pilot_split = build_pilot_split(full_split, pilot_fraction=0.05, seed=7)

print_split_summary(full_split, title="FULL WEAK DATASET (90/10 Train/dev split)")
print_split_summary(pilot_split, title="PILOT WEAK SPLIT (90/10 Train/dev split)")

tokenized_pilot = tokenize_split(pilot_split, tokenizer=tokenizer, max_length=512)

pilot_class_weights = build_class_weights(tokenized_pilot["train"])
pilot_sample_weights = build_sample_weights(tokenized_pilot["train"])

WEAK DATA CHECK
Examples      : 211269
Label counts  : {'yes': 196144, 'no': 15125}
Unique labels : ['no', 'yes']


Map:   0%|          | 0/211269 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/190142 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/21127 [00:00<?, ? examples/s]


FULL WEAK DATASET (90/10 Train/dev split)
train examples : 190142
train label counts : {'no': 13641, 'yes': 176501}

validation examples : 21127
validation label counts : {'no': 1484, 'yes': 19643}


PILOT WEAK SPLIT (90/10 Train/dev split)
train examples : 9507
train label counts : {'no': 677, 'yes': 8830}

validation examples : 1056
validation label counts : {'no': 67, 'yes': 989}



Map:   0%|          | 0/9507 [00:00<?, ? examples/s]

Map:   0%|          | 0/1056 [00:00<?, ? examples/s]

In [2]:
#First pilot stage

print_stage_header(
    "WEAK TRAINING PILOT",
    (
        "Cheap pilot on 10 percent of the PQA-A Dataset. "
        "Goal is to choose the best imbalance-handling method before full weak training."
    ),
)

pilot_configs = [
    {
        "run_name": "weak_pilot_plain",
        "learning_rate": 1e-5,
        "num_train_epochs": 4,
        "train_batch_size": 8,
        "eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "use_weighted_loss": False,
        "use_weighted_sampler": False,
    },
    {
        "run_name": "weak_pilot_weighted_loss",
        "learning_rate": 1e-5,
        "num_train_epochs": 4,
        "train_batch_size": 8,
        "eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "use_weighted_loss": True,
        "use_weighted_sampler": False,
    },
    {
        "run_name": "weak_pilot_weighted_sampler",
        "learning_rate": 1e-5,
        "num_train_epochs": 4,
        "train_batch_size": 8,
        "eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "use_weighted_loss": False,
        "use_weighted_sampler": True,
    },
    {
        "run_name": "weak_pilot_loss_plus_sampler",
        "learning_rate": 1e-5,
        "num_train_epochs": 4,
        "train_batch_size": 8,
        "eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "use_weighted_loss": True,
        "use_weighted_sampler": True,
    },
]

pilot_results = []

for cfg in pilot_configs:
    result = run_weak_experiment(
        tokenized_split=tokenized_pilot,
        tokenizer=tokenizer,
        class_weights=pilot_class_weights,
        sample_weights=pilot_sample_weights,
        seed=7,
        **cfg,
    )
    pilot_results.append(result)

print_results_table(pilot_results, "PILOT RESULTS")

best_pilot = choose_best_result(pilot_results, key="macro_f1")

print()
print("Best pilot setup")
print(best_pilot["run_name"])
print(best_pilot["final_metrics"])


WEAK TRAINING PILOT
Cheap pilot on 10 percent of the PQA-A Dataset. Goal is to choose the best imbalance-handling method before full weak training.



/Users/eduvieojoboh/Group-21-/BiomedBERTClinical/weak_stage_utils.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeakTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Yes
1,0.459500,0.160436,0.946970,0.688253,0.404255,0.972250
2,0.290300,0.099492,0.964962,0.837816,0.694215,0.981416
3,0.148700,0.168790,0.965909,0.827373,0.672727,0.982018
4,0.059800,0.200313,0.966856,0.848925,0.715447,0.982403


/Users/eduvieojoboh/Group-21-/BiomedBERTClinical/weak_stage_utils.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeakTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Yes
1,1.101800,0.354581,0.932765,0.749852,0.535948,0.963757
2,0.835500,0.366235,0.950758,0.806233,0.638889,0.973577
3,0.455500,0.793140,0.964962,0.832607,0.683761,0.981454
4,0.167700,0.724571,0.964015,0.837217,0.693548,0.980885


/Users/eduvieojoboh/Group-21-/BiomedBERTClinical/weak_stage_utils.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeakTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Yes
1,0.746200,0.227537,0.947917,0.801427,0.630872,0.971982
2,0.162000,0.271082,0.965909,0.852457,0.723077,0.981837
3,0.035300,0.397703,0.964962,0.824088,0.666667,0.981509
4,0.005100,0.425546,0.964962,0.821039,0.660550,0.981528


/Users/eduvieojoboh/Group-21-/BiomedBERTClinical/weak_stage_utils.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeakTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Yes
1,0.373200,0.408508,0.913826,0.727779,0.502732,0.952825
2,0.085200,1.171548,0.947917,0.791012,0.609929,0.972095
3,0.018400,1.605165,0.959280,0.817180,0.656000,0.978359
4,0.002900,2.047741,0.956439,0.779421,0.581818,0.977023



PILOT RESULTS
weak_pilot_plain | acc=0.9669 | macro_f1=0.8489 | f1_no=0.7154 | f1_yes=0.9824 | loss=False | sampler=False
weak_pilot_weighted_loss | acc=0.9640 | macro_f1=0.8372 | f1_no=0.6935 | f1_yes=0.9809 | loss=True | sampler=False
weak_pilot_weighted_sampler | acc=0.9659 | macro_f1=0.8525 | f1_no=0.7231 | f1_yes=0.9818 | loss=False | sampler=True
weak_pilot_loss_plus_sampler | acc=0.9593 | macro_f1=0.8172 | f1_no=0.6560 | f1_yes=0.9784 | loss=True | sampler=True

Best pilot setup
weak_pilot_weighted_sampler
{'accuracy': 0.9659090909090909, 'macro_f1': 0.8524567259178762, 'f1_no': 0.7230769230769231, 'f1_yes': 0.9818365287588294}


In [3]:
print_stage_header(
    "PILOT LR CHECK",
    (
        "Best method is now fixed. We now test one lower and one higher learning rate, "
        "then compare both against the existing best pilot with the base learning rate."
    ),
)

base_lr = best_pilot["learning_rate"]

extra_lr_candidates = [
    ("lr_low", 8e-6),
    ("lr_high", 1.2e-5),
]

extra_lr_results = []

for lr_name, lr_value in extra_lr_candidates:
    result = run_weak_experiment(
        run_name=f"weak_pilot_weighted_sampler_{lr_name}",
        tokenized_split=tokenized_pilot,
        tokenizer=tokenizer,
        learning_rate=lr_value,
        num_train_epochs=4,
        train_batch_size=8,
        eval_batch_size=8,
        gradient_accumulation_steps=best_pilot["gradient_accumulation_steps"],
        class_weights=pilot_class_weights,
        sample_weights=pilot_sample_weights,
        use_weighted_loss=best_pilot["use_weighted_loss"],
        use_weighted_sampler=best_pilot["use_weighted_sampler"],
        seed=7,
    )
    extra_lr_results.append(result)

lr_comparison_results = [best_pilot] + extra_lr_results

print_results_table(lr_comparison_results, "PILOT LR RESULTS")

best_lr_result = choose_best_result(lr_comparison_results, key="macro_f1")

print()
print("Best LR after local pilot check")
print(best_lr_result["run_name"])
print(best_lr_result["final_metrics"])
print(f"Chosen learning rate for full weak training: {best_lr_result['learning_rate']}")


PILOT LR CHECK
Best method is now fixed. We now test one lower and one higher learning rate, then compare both against the existing best pilot with the base learning rate.



/Users/eduvieojoboh/Group-21-/BiomedBERTClinical/weak_stage_utils.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeakTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Yes
1,0.836000,0.184868,0.937500,0.768860,0.571429,0.966292
2,0.202400,0.440032,0.939394,0.790785,0.614458,0.967112
3,0.047200,0.461919,0.953598,0.788495,0.601626,0.975365
4,0.012500,0.512498,0.954545,0.773714,0.571429,0.976000


/Users/eduvieojoboh/Group-21-/BiomedBERTClinical/weak_stage_utils.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeakTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Yes
1,0.735600,0.206457,0.941288,0.792745,0.617284,0.968205
2,0.144800,0.309788,0.960227,0.837188,0.695652,0.978723
3,0.028100,0.426908,0.960227,0.802000,0.625000,0.979000
4,0.009800,0.485096,0.960227,0.791418,0.603774,0.979063



PILOT LR RESULTS
weak_pilot_weighted_sampler | acc=0.9659 | macro_f1=0.8525 | f1_no=0.7231 | f1_yes=0.9818 | loss=False | sampler=True
weak_pilot_weighted_sampler_lr_low | acc=0.9394 | macro_f1=0.7908 | f1_no=0.6145 | f1_yes=0.9671 | loss=False | sampler=True
weak_pilot_weighted_sampler_lr_high | acc=0.9602 | macro_f1=0.8372 | f1_no=0.6957 | f1_yes=0.9787 | loss=False | sampler=True

Best LR after local pilot check
weak_pilot_weighted_sampler
{'accuracy': 0.9659090909090909, 'macro_f1': 0.8524567259178762, 'f1_no': 0.7230769230769231, 'f1_yes': 0.9818365287588294}
Chosen learning rate for full weak training: 1e-05


In [6]:
# Now full stage
import importlib
import weak_stage_utils
importlib.reload(weak_stage_utils)
from weak_stage_utils import *

full_split = load_dataset_split(full_split_save_path)

tokenized_full = tokenize_split(full_split, tokenizer=tokenizer, max_length=512)

full_class_weights = build_class_weights(tokenized_full["train"])
full_sample_weights = build_sample_weights(tokenized_full["train"])

print_stage_header(
    "FULL DATASET WEAK TRAINING",
    (
        "Train on the full PQA-A dataset using the best pilot config. "
        "This exported model becomes the reusable warm-start checkpoint for all Stage 2 ablations."
    ),
)

full_run_result = run_weak_experiment(
    run_name="weak_full_best_method",
    tokenized_split=tokenized_full,
    tokenizer=tokenizer,
    learning_rate=best_lr_result["learning_rate"],
    num_train_epochs=6,
    train_batch_size=8,
    eval_batch_size=8,
    gradient_accumulation_steps=best_pilot["gradient_accumulation_steps"],
    class_weights=full_class_weights,
    sample_weights=full_sample_weights,
    use_weighted_loss=best_pilot["use_weighted_loss"],
    use_weighted_sampler=best_pilot["use_weighted_sampler"],
    seed=7,
)

print(full_run_result["final_metrics"])

Map:   0%|          | 0/190142 [00:00<?, ? examples/s]

Map:   0%|          | 0/21127 [00:00<?, ? examples/s]


FULL DATASET WEAK TRAINING
Train on the full PQA-A dataset using the best pilot config. This exported model becomes the reusable warm-start checkpoint for all Stage 2 ablations.



/Users/eduvieojoboh/Group-21-/BiomedBERTClinical/weak_stage_utils.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeakTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Yes
1,0.252400,0.166852,0.973494,0.905763,0.825871,0.985655
2,0.066700,0.173664,0.978984,0.916034,0.843331,0.988737
3,0.031500,0.196641,0.977186,0.914939,0.842174,0.987704
4,0.015500,0.256066,0.980972,0.921979,0.854136,0.989822
5,0.007500,0.304057,0.980594,0.919392,0.849154,0.989630
6,0.002100,0.335152,0.979505,0.914193,0.839332,0.989054


{'accuracy': 0.9809722156482227, 'macro_f1': 0.9219793495450634, 'f1_no': 0.8541364296081277, 'f1_yes': 0.9898222694819991}


In [7]:
# Exporting best config to use for downstream fine-tuning
export_metadata = {
    "stage": "stage1_binary_weak_warmstart",
    "model_id": model_id,
    "label_to_id": label_to_id,
    "id_to_label": id_to_label,
    "max_length": 512,
    "truncation": "only_second",
    "split": "90_train_10_validation",
    "pilot_fraction": 0.05,
    "best_method_from_pilot": {
        "use_weighted_loss": best_pilot["use_weighted_loss"],
        "use_weighted_sampler": best_pilot["use_weighted_sampler"],
        "learning_rate": best_lr_result["learning_rate"],
        "gradient_accumulation_steps": best_pilot["gradient_accumulation_steps"],
    },
    "full_run_metrics": full_run_result["final_metrics"],
    "best_checkpoint": full_run_result["best_checkpoint"],
}

export_best_weak_model(
    best_checkpoint_path=full_run_result["best_checkpoint"],
    tokenizer=tokenizer,
    export_dir="./artifacts/stage1_binary_warmstart_best",
    metadata=export_metadata,
)

# Delete suboptimal runs
delete_non_best_run_dirs(
    results=pilot_results + extra_lr_results + [full_run_result],
    best_run_name="weak_full_best_method",
    runs_root="./weak_runs",
)


